# A Logic Puzzle

The following exercise is taken from the book 
<a href="https://www.amazon.de/Logeleien-Zweistein-ihren-Antworten-Wegner/dp/B006YF0VUE">"99 Logeleien von Zweistein"</a>.
This book has been published 1968.  It is written by 
<a href="http://de.wikipedia.org/wiki/Thomas_von_Randow">Thomas von Randow</a>.

---
The gentlemen Amann, Bemann, Cemann and Demann are called - not necessarily in the same order - by their first names Erich, Fritz, Gustav and Heiner. They are all married to exactly one woman. We also know the following about them and their wives:

- Either Amann's first name is Heiner, or Bemann's wife is Inge.
- If Cemann is married to Josefa, then - **and only in this case** - Klara's husband is **not** called Fritz.
- If Josefa's husband is **not** called Erich, then Inge is married to Fritz.
- If Luise's husband is called Fritz, then Klara's husband's first name is **not** Gustav.
- If the wife of Fritz is called Inge, then Erich is **not** married to Josefa.
- If Fritz is **not** married to Luise, then Gustav's wife's name is Klara.
- Either Demann is married to Luise, or Cemann is called Gustav.

*What are the full fullnames of these gentlemen, and what are their wives' first names?*

---

We are going to solve this problem by coding it in propositional logic and we will solve the resulting set of clauses using the Davis-Putnam algorithm.  In order to code the problem, we will use the following propositional variables:

- $\texttt{MaleName<}x\texttt{,}z\texttt{>}$ for any male first name $x$ and any surname $z$ expresses
  that the gentleman with first name $x$ has surname $z$.
- $\texttt{Married<}x\texttt{,}y\texttt{>}$ for any male first name $x$ and any female first name $y$ expresses
  that the gentleman with first name $x$ is married to the lady with first name $y$.
- $\texttt{FemaleName<}x\texttt{,}z\texttt{>}$ for any female first name $x$ and any last name $z$ expresses
  that the lady with first name $x$ has surname $z$.

We are using the symbols $\texttt{<}$ and $\texttt{>}$ as part of the propositional variables because we want to show the structure of these variables and the parser for propositional logic accepts these symbols as part of propositional variables.

We load the class `RecursiveSet`, the record classes representing formulas and literals, and the implementation of the Davis-Putnam algorithm, which provides the function `solve`.

In [ ]:
%load ../lib/RecursiveSet.jsh
%load Formula.jsh
%load 06-Davis-Putnam.jsh

In [ ]:
@SafeVarargs
<T> RecursiveSet<T> set(T... elements) {
    return new RecursiveSet<>(elements);
}

In [ ]:
var FirstMale   = set("Erich", "Fritz", "Gustav", "Heiner");
var FirstFemale = set("Inge",  "Josefa", "Klara", "Luise");
var SurNames    = set("Amann", "Bemann", "Cemann", "Demann");

Instead of using the parser for propositional logic, we define a few helper functions that directly create clauses.  A literal is either a variable `new Var(p)` or a negated variable `new Neg(p)`.  The function `pos(p)` returns the positive literal $p$, while `not(p)` returns the negative literal $\neg p$.  The function `complement(l)` computes the complement of the literal `l`.

In [ ]:
Literal complement(Literal l) {
    return switch (l) {
        case Var(var p) -> new Neg(p);
        case Neg(var p) -> new Var(p);
        default -> throw new IllegalArgumentException("unknown literal: " + l);
    };
}

Literal pos(String p) {
    return new Var(p);
}

Literal not(String p) {
    return new Neg(p);
}

The function `makeVar(f, x, y)` creates a propositional variable of the form `f<x,y>`.

In [ ]:
String makeVar(String f, String x, String y) {
    return f + "<" + x + "," + y + ">";
}

In [ ]:
makeVar("Married", "Heiner", "Klara")

Given a set of propositional variables $S$, the function `atMostOne(S)` computes a set of clauses expressing the fact that at most one of the variables of $S$ is `True`.

In [ ]:
RecursiveSet<RecursiveSet<Literal>> atMostOne(RecursiveSet<String> S) {
    return flatMap(S, p -> S.filterMap(q -> !p.equals(q),
                                       q -> set(not(p), not(q))));
}

Given a set of propositional variables $S$, the function `atLeastOne(S)` computes a set of clauses expressing the fact that at least one of the variables of $S$ is `True`.

In [ ]:
RecursiveSet<RecursiveSet<Literal>> atLeastOne(RecursiveSet<String> S) {
    return set(S.map(p -> pos(p)));
}

$S$ is a set of propositional variables. The expression `exactlyOne(S)` creates a set of clauses.  This set expresses the fact that exactly one of the variables in the set $S$ is true.

In [ ]:
RecursiveSet<RecursiveSet<Literal>> exactlyOne(RecursiveSet<String> S) {
    return atMostOne(S).union(atLeastOne(S));
}

In [ ]:
exactlyOne(set("a", "b", "c"))

For two sets $A$ and $B$ that have the same number of elements and a function symbol $f$, the procedure `bijective(A, B)` computes a set of clauses that is equivalent to the formula
$$   \bigl(\forall x \in A: \exists! y \in B: f\langle x, y\rangle\bigr) \wedge
     \bigl(\forall y \in B: \exists! x \in A: f\langle x, y\rangle\bigr)
$$
Here the expression $f\langle x,y\rangle$ is the name of a propositional variable and the expression $\exists!x:p(x)$ is to be read as "There exists exactly one $x$ such that $p(x)$ holds".

In [ ]:
RecursiveSet<RecursiveSet<Literal>> bijective(RecursiveSet<String> A, RecursiveSet<String> B, String f) {
    RecursiveSet<RecursiveSet<Literal>> clauses = set();
    // your code here
    return clauses;
}

In [ ]:
bijective(set("a", "b"), set("x", "y"), "f")

We introduce standard logical connectives (`implies`, `exclusiveOr`, `equivalence`, `impliesAnd`) as helper functions that directly yield sets of clauses, bypassing the need for a parser.

In [ ]:
RecursiveSet<RecursiveSet<Literal>> implies(Literal a, Literal b) {
    // a → b ≡ ¬a ∨ b
    return set(set(complement(a), b));
}

RecursiveSet<RecursiveSet<Literal>> exclusiveOr(Literal a, Literal b) {
    // a ↔ ¬b ≡ (a ∨ b) ∧ (¬a ∨ ¬b)
    return set(set(a, b), set(complement(a), complement(b)));
}

RecursiveSet<RecursiveSet<Literal>> equivalence(Literal a, Literal b) {
    // a ↔ b ≡ (¬a ∨ b) ∧ (a ∨ ¬b)
    return set(set(complement(a), b), set(a, complement(b)));
}

RecursiveSet<RecursiveSet<Literal>> impliesAnd(Literal a, Literal b, Literal c) {
    // a ∧ b → c ≡ ¬a ∨ ¬b ∨ c
    return set(set(complement(a), complement(b), c));
}

In [ ]:
exclusiveOr(pos("p"), pos("q"))

Given a set of male first names `FirstMale`, a set of female first names `FirstFemale`, and a set of surnames `Surnames`,
the function `consistentNames(FirstMale, FirstFemale, Surnames)` returns a set of clauses that ensures that if the man
`x` has the surname `z` and the woman `y` also has the surname `z`, then `x` and `y` have to be married.

In [ ]:
RecursiveSet<RecursiveSet<Literal>> consistentNames(RecursiveSet<String> FirstMale,
                                                    RecursiveSet<String> FirstFemale,
                                                    RecursiveSet<String> SurNames) {
    return flatMap(FirstMale, x ->
           flatMap(FirstFemale, y ->
           flatMap(SurNames, z -> {
               var a = pos(makeVar("MaleName",   x, z));
               var b = pos(makeVar("FemaleName", y, z));
               var c = pos(makeVar("Married",    x, y));
               return impliesAnd(a, b, c);
           })));
}

In [ ]:
consistentNames(FirstMale, FirstFemale, SurNames)

Now we translate the constraints given in the text into clauses using our helper functions.

In [ ]:
RecursiveSet<RecursiveSet<Literal>> computeClauses(RecursiveSet<String> FirstMale,
                                                   RecursiveSet<String> FirstFemale,
                                                   RecursiveSet<String> SurNames) {
    RecursiveSet<RecursiveSet<Literal>> clauses = set();
    // Jedem männlichen Vornamen ist genau ein Nachname zugeordnet und umgekehrt.
    // your code here
    // Jeder Mann ist mit genau einer Frau verheiratet und umgekehrt.
    // your code here
    // Jede Frau hat genau einen Nachnamen.
    // your code here
    // Die Namen sind konsistent.
    // your code here
    // Entweder ist Amanns Vorname Heiner, oder Bemanns Frau heisst Inge.
    // your code here
    // Wenn Cemann mit Josefa verheiratet ist, dann – und nur in diesem Falle – heisst Klaras Mann nicht Fritz.
    // your code here
    // Wenn Josefas Mann nicht Erich heisst, dann ist Inge mit Fritz verheiratet.
    // your code here
    // Wenn Luises Mann Fritz heisst, dann ist der Vorname von Klaras Mann nicht Gustav.
    // your code here
    // Wenn die Frau von Fritz Inge heisst, dann ist Erich nicht mit Josefa verheiratet.
    // your code here
    // Wenn Fritz nicht mit Luise verheiratet ist, dann heisst Gustavs Frau Klara.
    // your code here
    // Entweder ist Demann mit Luise verheiratet, oder Cemann heisst Gustav.
    // your code here
    return clauses;
}

In [ ]:
var Clauses = computeClauses(FirstMale, FirstFemale, SurNames);
Clauses

There are 242 different clauses.

In [ ]:
Clauses.size()

In [ ]:
RecursiveSet<RecursiveSet<Literal>> computeSolution(RecursiveSet<String> FirstMale,
                                                    RecursiveSet<String> FirstFemale,
                                                    RecursiveSet<String> SurNames) {
    var systemClauses = computeClauses(FirstMale, FirstFemale, SurNames);
    return solve(systemClauses);
}

In [ ]:
var Solution = computeSolution(FirstMale, FirstFemale, SurNames);

The function `onlyPositive` returns the names of those variables that are `true` in the given solution.  It uses the function `arb`, which returns a random element of a set.  This function is defined in the file `06-Davis-Putnam.jsh`.  We must not define it again here:  In JShell, redefining a method resets all variables whose values have been computed using this method.  Hence, redefining `arb` would reset the variable `Solution` to `null`.

In [ ]:
RecursiveSet<String> onlyPositive(RecursiveSet<RecursiveSet<Literal>> Solution) {
    RecursiveSet<String> result = set();
    for (var clause : Solution) {
        if (arb(clause) instanceof Var(var p)) {
            result.add(p);
        }
    }
    return result;
}

In [ ]:
onlyPositive(Solution)

The function `extractFirst` takes a variable name of the form `f<x,y>` and returns `x`, while `extractSecond` returns `y`.

In [ ]:
String extractFirst(String s) {
    return s.substring(s.indexOf('<') + 1, s.indexOf(','));
}

String extractSecond(String s) {
    return s.substring(s.indexOf(',') + 1, s.indexOf('>'));
}

In [ ]:
void displaySolution(RecursiveSet<RecursiveSet<Literal>> Solution) {
    Map<String, String> married = new TreeMap<>();
    Map<String, String> names   = new TreeMap<>();
    for (var unit : Solution) {
        for (var l : unit) {
            if (l instanceof Var(var p)) {
                if (p.startsWith("Married")) {
                    married.put(extractFirst(p), extractSecond(p));
                } else if (p.startsWith("MaleName")) {
                    names.put(extractFirst(p), extractSecond(p));
                }
            }
        }
    }
    for (var x : married.keySet()) {
        System.out.println(x + " " + names.get(x) + " is married to " + married.get(x) + ".");
    }
}

In [ ]:
displaySolution(Solution);

## Checking the Uniqueness of the Solution

Given a set of unit clauses $U$, the function `negateSolution(U)` returns a clause that is the negation of the set $U$.

In [ ]:
RecursiveSet<Literal> negateSolution(RecursiveSet<RecursiveSet<Literal>> UnitClauses) {
    return UnitClauses.map(unit -> complement(arb(unit)));
}

In [ ]:
negateSolution(set(set(pos("a")), set(not("b"))))

In [ ]:
void checkUniqueness(RecursiveSet<RecursiveSet<Literal>> Solution,
                     RecursiveSet<RecursiveSet<Literal>> baseClauses) {
    var negation         = negateSolution(Solution);
    var augmentedClauses = baseClauses.union(set(negation));
    var alternative      = solve(augmentedClauses);
    RecursiveSet<Literal> emptyClause = set();
    // If alternative contains the empty clause {}, the augmented clauses are inconsistent
    if (alternative.contains(emptyClause) && alternative.size() == 1) {
        System.out.println("Well done: The solution is unique!");
    } else {
        System.out.println("ERROR: The solution is not unique!");
    }
}

In [ ]:
checkUniqueness(Solution, Clauses);